# Saudi Plate OCR — Train on Kaggle

Same pipeline as the Colab version, adapted for Kaggle (~30 GPU hrs/week).

## ONE-TIME SETUP (do this before running cells)
In the right-hand panel:
1. **Session options → Accelerator → `GPU T4 x2`** (or P100).
2. **Session options → Internet → On** (needed for pip/clone/download).
3. **Add-ons → Secrets → Add a new secret**: name `GH_TOKEN`, value = your GitHub token (repo read). Toggle it **Attached** to this notebook.

Then run cells top to bottom.


## 1. Install
Paddle 2.6.2 (CUDA 11.8) + PaddleOCR 2.7 + NumPy<2. ~3-4 min.


In [ ]:
!apt-get -qq update && apt-get -qq -y install fonts-noto-core fonts-kacst fonts-hosny-amiri fonts-dejavu-core > /dev/null 2>&1
!pip install -q paddlepaddle-gpu==2.6.2 --extra-index-url https://www.paddlepaddle.org.cn/packages/stable/cu118/
!rm -rf PaddleOCR
!git clone -q --depth 1 -b release/2.7 https://github.com/PaddlePaddle/PaddleOCR.git
!pip install -q -r PaddleOCR/requirements.txt
!pip install -q 'paddle2onnx==1.1.0' onnxruntime
!pip install -q 'numpy<2'
import paddle
print('paddle', paddle.__version__, '| GPU:', paddle.device.is_compiled_with_cuda())
assert paddle.device.is_compiled_with_cuda(), 'No GPU — set Accelerator to GPU in Session options'


## 2. Get our code from GitHub
Reads the `GH_TOKEN` secret (Kaggle's secret system, not Colab's).


In [ ]:
from kaggle_secrets import UserSecretsClient
import subprocess
TOKEN = UserSecretsClient().get_secret('GH_TOKEN').strip()
subprocess.run('rm -rf saudi-plate-ocr', shell=True)
subprocess.run(['git','clone','-q',f'https://{TOKEN}@github.com/Tanzimalam1454/saudi-plate-ocr.git'])
for f in ['plate_spec.py','generate_plates.py','saudi_rec_config.yml']:
    subprocess.run(['cp', f'saudi-plate-ocr/{f}', '.'])
import os; assert all(os.path.exists(f) for f in ['plate_spec.py','generate_plates.py','saudi_rec_config.yml'])
print('files in place')


## 3. Generate synthetic data (multi-core, ~5 min)
8,000 plates -> 16,000 labeled half-crops, train/val split.


In [ ]:
!rm -rf dataset
!python generate_plates.py --count 8000 --out dataset --montage
!wc -l dataset/train_label.txt dataset/val_label.txt
from IPython.display import Image as IPImage; IPImage('dataset/_montage.jpg', width=900)


## 4. Download pretrained Arabic recognizer


In [ ]:
import os, glob
if not glob.glob('pretrain/arabic_PP-OCRv*_rec_train'):
    os.makedirs('pretrain', exist_ok=True)
    !cd pretrain && wget -q https://paddleocr.bj.bcebos.com/PP-OCRv4/multilingual/arabic_PP-OCRv4_rec_train.tar && tar xf arabic_PP-OCRv4_rec_train.tar
!ls pretrain


## 5. Train (saves best model each epoch)
Watch the `best metric, acc:` line. Stop when it plateaus — best model is kept in
`output/saudi_rec/best_accuracy`.


In [ ]:
import glob, re
pre = glob.glob('pretrain/arabic_PP-OCRv*_rec_train')[0] + '/best_accuracy'
n_train = sum(1 for l in open('dataset/train_label.txt') if l.strip())
iters = max(1, n_train // 64)
cfg = open('saudi_rec_config.yml').read()
cfg = re.sub(r'pretrained_model:.*', f'pretrained_model: ./{pre}', cfg, count=1)
cfg = re.sub(r'eval_batch_step:.*', f'eval_batch_step: [0, {iters}]', cfg, count=1)
open('saudi_rec_config.yml','w').write(cfg)
print(f'{n_train} train crops -> eval + save best every {iters} iters (per epoch)')
!python PaddleOCR/tools/train.py -c saudi_rec_config.yml \
  -o Global.epoch_num=12 Optimizer.lr.warmup_epoch=1 \
     Train.loader.batch_size_per_card=64 Eval.loader.batch_size_per_card=64


## 6. Test on your own images (no export)
On Kaggle you can't pop up a file picker. Two easy ways to get your plate images in:

**A)** Right panel → **Input → + Add Input → Upload** a folder/zip of your CROPPED plate images.
It mounts read-only at `/kaggle/input/<your-dataset-name>/`.

**B)** Or just put images in `/kaggle/working/myplates/` via the file browser.

Set `SRC` below to wherever your images are, then run.


In [ ]:
SRC = '/kaggle/working/myplates'   # <-- change to your /kaggle/input/<name> folder if you added one

from PIL import Image
import numpy as np, os, glob
from plate_spec import split_saudi_plate
os.makedirs('myhalves', exist_ok=True)
imgs = [f for f in glob.glob(f'{SRC}/*') if f.lower().endswith(('.png','.jpg','.jpeg'))]
assert imgs, f'no images found in {SRC} — add them first (see the cell above)'
for p in imgs:
    img = np.array(Image.open(p).convert('RGB'))[:, :, ::-1]
    top, bottom = split_saudi_plate(img)
    stem = os.path.splitext(os.path.basename(p))[0]
    Image.fromarray(bottom[:, :, ::-1]).save(f'myhalves/{stem}__EN.jpg')
    Image.fromarray(top[:, :, ::-1]).save(f'myhalves/{stem}__AR.jpg')
print('halves:', os.listdir('myhalves'))
!python PaddleOCR/tools/infer_rec.py -c saudi_rec_config.yml \
  -o Global.checkpoints=./output/saudi_rec/best_accuracy Global.infer_img=./myhalves


---
## 7. (LATER) Export to ONNX
Only after you're happy with the readings.


In [ ]:
!python PaddleOCR/tools/export_model.py -c saudi_rec_config.yml \
  -o Global.checkpoints=./output/saudi_rec/best_accuracy Global.save_inference_dir=./inference/saudi_rec
import os
mfile = 'inference.json' if os.path.exists('inference/saudi_rec/inference.json') else 'inference.pdmodel'
!paddle2onnx --model_dir ./inference/saudi_rec --model_filename {mfile} \
  --params_filename inference.pdiparams --save_file ./inference/saudi_rec.onnx --opset_version 13
!ls -la inference
